# ONE-CELL repair: rebuild hist_psl and update climx-cache-extras

Everything is automatic — the notebook derives the manifests from the attached
datasets, so there are no manual steps to get wrong.

**Setup (once, before running):** right panel → **Data → Add Input → Datasets** →
add **`climx-cache-ssp126`** AND **`climx-cache-extras`**. Accelerator: NONE (CPU).

Expected: everything skips except `=== write hist window psl ===`, then a real
extras version upload. ~20-30 min total.

In [ ]:
import os, subprocess, sys, pathlib, json
repo = "/kaggle/working/climx"
if os.path.isdir(repo):
    r = subprocess.run(["git", "-C", repo, "pull", "--ff-only"], capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/nonewanit/climx-emulator.git", repo], check=True)
os.chdir(repo)
sys.path.insert(0, str(pathlib.Path(repo) / "src"))
subprocess.run(["find", repo, "-name", "__pycache__", "-type", "d",
                "-exec", "rm", "-rf", "{}", "+"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
r = subprocess.run(["git", "-C", repo, "log", "-1", "--oneline"], capture_output=True, text=True)
print("HEAD:", r.stdout.strip(), "— expect 0ffde97 or newer")

# --- locate the attached cache datasets ---
dirs = {p.name: str(p) for p in pathlib.Path("/kaggle/input").rglob("climx-cache*") if p.is_dir()}
print("attached:", sorted(dirs))
assert "climx-cache-ssp126" in dirs, "attach climx-cache-ssp126 as an input"
assert "climx-cache-extras" in dirs, "attach climx-cache-extras as an input"
os.environ["CLIMX_RECOVER_PKG_DIR"] = dirs["climx-cache-ssp126"]
os.environ["CLIMX_RECOVER_EXTRAS_DIR"] = dirs["climx-cache-extras"]

# --- rebuild the manifests from the mounts (ground truth) ---
def pkg_stems(d):
    stems = []
    for q in sorted(pathlib.Path(d).iterdir()):
        if q.name.endswith(".zarrpkg"):
            stems.append(q.name[:-8].rsplit(".zarr", 1)[0])
        elif q.name == "X_test.nc":
            stems.append(q.name)
    return stems
data_dir = pathlib.Path(repo) / "data"
data_dir.mkdir(parents=True, exist_ok=True)
# scenario manifest: everything in the ssp126 dataset is correct and uploaded
json.dump({"nonewanitchollakit/climx-cache-ssp126": pkg_stems(dirs["climx-cache-ssp126"])},
          open(data_dir / "full_cache_manifest_ssp126.json", "w"), indent=2)
# extras manifest: everything EXCEPT hist_psl (the broken one we rebuild)
extras_keep = [s for s in pkg_stems(dirs["climx-cache-extras"]) if s != "hist_psl"]
json.dump({"nonewanitchollakit/climx-cache-extras": extras_keep},
          open(data_dir / "full_cache_manifest_extras.json", "w"), indent=2)
print("manifests rebuilt from the mounts; extras keeps:", extras_keep)
print("ready — run the next cell")

## Run the repair

In [ ]:
%run scripts/full_cache.py --scenarios ssp126

## What to expect

1. `=== pass B (ssp126) ===` (lazy build, seconds)
2. scenario writes **skipped**; scenario upload prints `nothing new to upload`
3. `X_test.nc` skipped
4. `=== write hist window psl ===` → `OK hist psl (0.6xx GB)`
5. `=== upload batch … -> nonewanitchollakit/climx-cache-extras ===` with a real duration
6. `CACHE_UPLOAD_OK (ssp126)`

Paste the log to None for verification.